# 02 — Dataset Split
Split raw images into train / val / test (70 / 15 / 15). Run once only.

In [1]:
import shutil
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

RAW_DIR  = Path('../data/raw')
OUT_DIR  = Path('../data/processed')
CLASSES  = ['Bacterial_Blight', 'Blast', 'Brown_Spot', 'Healthy_Rice_Leaf', 'Rice_Hispa', 'Tungro']
SEED     = 42


## Create output folders

In [2]:
for split in ('train', 'val', 'test'):
    for cls in CLASSES:
        (OUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)
print('Folders created.')


Folders created.


## Copy images into splits

In [3]:
summary = {}

for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    images  = sorted([
        p for ext in ('*.jpg','*.jpeg','*.JPG','*.JPEG','*.png')
        for p in cls_dir.glob(ext)
    ])

    train_imgs, temp = train_test_split(images, test_size=0.30,
                                         random_state=SEED, shuffle=True)
    val_imgs, test_imgs = train_test_split(temp,  test_size=0.50,
                                            random_state=SEED, shuffle=True)

    for split_name, split_imgs in [('train', train_imgs),
                                    ('val',   val_imgs),
                                    ('test',  test_imgs)]:
        for src in split_imgs:
            dst = OUT_DIR / split_name / cls / src.name
            shutil.copy2(src, dst)

    summary[cls] = dict(total=len(images),
                         train=len(train_imgs),
                         val=len(val_imgs),
                         test=len(test_imgs))

print(f'{"Class":<20} {"Total":>6} {"Train":>6} {"Val":>5} {"Test":>5}')
print('-' * 46)
totals = dict(total=0, train=0, val=0, test=0)
for cls, c in summary.items():
    print(f'{cls:<20} {c["total"]:>6} {c["train"]:>6} {c["val"]:>5} {c["test"]:>5}')
    for k in totals: totals[k] += c[k]
print('-' * 46)
print(f'{"TOTAL":<20} {totals["total"]:>6} {totals["train"]:>6} {totals["val"]:>5} {totals["test"]:>5}')


Class                 Total  Train   Val  Test
----------------------------------------------
Bacterial_Blight       1584   1108   238   238
Blast                  1464   1024   220   220
Brown_Spot             1751   1225   263   263
Healthy_Rice_Leaf       653    457    98    98
Rice_Hispa             1594   1115   239   240
Tungro                 1486   1040   223   223
----------------------------------------------
TOTAL                  8532   5969  1281  1282


## Verify split

In [4]:
for split in ('train', 'val', 'test'):
    n = sum(len(list((OUT_DIR / split / cls).glob('*.*'))) for cls in CLASSES)
    print(f'{split:<6}: {n:,} images')
print('\nSplit complete.')


train : 10,771 images
val   : 2,457 images
test  : 2,463 images

Split complete.
